# Lesson 1 — Try working policies

## The idea before the implementation

**From a scene to a robot action.** A camera frame does not tell a robot which command to send. A policy learns that connection from demonstrations. In this course, each policy generates a sequence of seven-value controller commands called an action chunk. The controller applies selected actions one at a time, and new observations let the policy correct its next prediction.

**Where you work.** You are at the complete-system level. ACT predicts a chunk in one forward pass. Diffusion Policy generates an action window through repeated noise-removal updates. SmolVLA uses images, an instruction, and state to guide an action expert. These are three separate policies; they do not form a chain in which one repairs another.

**Pause and explain.** While you watch a rollout, identify the observation, the predicted motion, and the next correction. A video can suggest a failure mode, but one successful video does not establish a success rate.

Keep [the concept guide](../concepts/control-and-learning.md) open beside this notebook for the full system diagrams and lesson map.

Start with a working system. Run ACT, SmolVLA, and Diffusion Policy on the same task. Change a starting state or the SmolVLA prompt. Watch the video and record what you see.

This unit is a playground. It does not ask you to implement model code yet. Later units will replace one part of this system at a time.

**Time:** 15 minutes to inspect saved runs; 30+ minutes for live trials. Each model has a separate live cell. Run the cell to load its local checkpoint and execute one development episode. It does not download weights or data.

The local prepared weights are different recipes. The table below is saved evidence, not a new run. Compare actions and behavior as a first look. Lesson 22 adds a controlled, paired evaluation.

# Before you code

This course assumes strong Python skills and basic machine-learning knowledge. We explain PyTorch when it first appears. Read each shape, unit, mask, and parameter definition before you run the check. The reference copy gives a complete solution; the learner copy leaves the implementation for you.

All three policies use an observation to produce actions, but their internal models differ. ACT predicts a fixed action sequence in one pass. Diffusion Policy begins with noise and removes noise over several steps. SmolVLA combines visual and language inputs in a vision-language-action model. Here you will run each supplied policy on LIBERO before you study or change its internals. The first comparison uses the same task and development states where local runs exist. Treat each video as one observed rollout, not as a measure of general ability.

The next lessons focus on ACT. We will first write the control loop around a supplied policy, then inspect its data, training objective, and model blocks. This order follows a real deployment: a trained model is useful only when its inputs and outputs match the robot interface.


In [1]:
import os
import sys
from pathlib import Path


def find_workspace(start: str | Path | None = None) -> Path:
    current = Path(start or Path.cwd()).resolve()
    if current.is_file():
        current = current.parent
    for folder in (current, *current.parents):
        if (folder / "manifests" / "act_references.yaml").is_file():
            return folder
    raise FileNotFoundError("Open this notebook inside the VLA course workspace.")


WORKSPACE = find_workspace()
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))

# Use local caches and prevent model or data downloads when importing libraries.
HF_CACHE = WORKSPACE / ".local-data" / "hf"
(HF_CACHE / "hub").mkdir(parents=True, exist_ok=True)
(HF_CACHE / "datasets").mkdir(parents=True, exist_ok=True)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE / "hub")
os.environ["HF_DATASETS_CACHE"] = str(HF_CACHE / "datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

print("Workspace:", WORKSPACE)

Workspace: /home/michal/code/vlas_tutorial


## Saved evidence

Read the table as prior recorded results. ACT and SmolVLA used all 20 development states. The Diffusion row used states 0–4. These counts do not form a direct comparison because the evaluated state sets differ.

In [2]:
import json

from IPython.display import Video, display

SAVED_RUNS = {
    "ACT": (".local-data/runs/act-bowl-plate-seeded-dev-all/summary.json", "19 / 20"),
    "SmolVLA": (".local-data/runs/smolvla-published-n10-seeded-dev-all/summary.json", "16 / 20"),
    "Diffusion Policy": (
        ".local-data/runs/diffusion-2000-seeded-dev-20step-timed-0-4/summary.json",
        "2 / 5",
    ),
}
for label, (relative_path, successes) in SAVED_RUNS.items():
    path = WORKSPACE / relative_path
    if path.is_file():
        record = json.loads(path.read_text())
        print(
            f"Saved evidence: {label}: {successes} successes; states {record['state_ids'][0]}–{record['state_ids'][-1]}; {path.relative_to(WORKSPACE)}"
        )
    else:
        print(f"Saved evidence is not in this checkout: {label}: {relative_path}")

# This is one saved ACT video. It does not come from the live cell below.
saved_video = WORKSPACE / ".local-data/runs/act-bowl-plate-probe-200/rollout-state-000.mp4"
if saved_video.is_file():
    display(Video(str(saved_video), embed=True, mimetype="video/mp4", html_attributes="controls"))

Saved evidence: ACT: 19 / 20 successes; states 0–19; .local-data/runs/act-bowl-plate-seeded-dev-all/summary.json
Saved evidence: SmolVLA: 16 / 20 successes; states 0–19; .local-data/runs/smolvla-published-n10-seeded-dev-all/summary.json
Saved evidence: Diffusion Policy: 2 / 5 successes; states 0–4; .local-data/runs/diffusion-2000-seeded-dev-20step-timed-0-4/summary.json


## Choose a case

Try state 0, then another state from 0 to 9. These are development states for exploration. Change the prompt only to test SmolVLA. Keep a short note of the policy, state, prompt, outcome, and an observation about its motion.

In [3]:
TASK = "bowl_plate"
STATE_IDS = [0]  # Try values from 0 to 9.
# METHODS = ["act", "smolvla_published", "diffusion"]
METHODS = ["act", "diffusion", "smolvla_published"]
SMOLVLA_PROMPT = "put the grey bowl on the plate"
# SMOLVLA_PROMPT = "put a bowl on a plate"
MAX_STEPS = 260

assert STATE_IDS and all(0 <= state <= 9 for state in STATE_IDS), "Use development states 0–9 here."
assert set(METHODS) <= {"act", "smolvla_published", "smolvla_learner", "diffusion"}
assert 1 <= MAX_STEPS <= 300

## Run prepared policies

The three full LIBERO runs below use separate cells and display one video per policy. Run each cell to replay that policy. The loader checks each saved checkpoint against its manifest. The simulator records a video and result file under `.local-data/course-runs`. A repeated run uses a new output folder.

In [6]:
import gc
import json
from uuid import uuid4

import torch
from IPython.display import Video, display

from vla_lab.evaluation.demo import encode_rgb_video
from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.interfaces import ActionPolicy, TensorBatch
from vla_lab.teaching.live import load_prepared_policy


def run_one(method: str, state_id: int, prompt=SMOLVLA_PROMPT) -> dict[str, object]:
    """Run one prepared policy on one fixed LIBERO development state."""
    task = "bowl_plate"
    max_steps = 160
    output = WORKSPACE / ".local-data/course-runs/unit01" / f"{method}-{state_id}-{uuid4().hex[:8]}"
    output.mkdir(parents=True, exist_ok=False)
    frames = []
    try:
        policy, preprocessor, postprocessor, recipe = load_prepared_policy(
            method, WORKSPACE, task_alias=task
        )
    except Exception as error:
        row = {
            "method": method,
            "state": state_id,
            "success": False,
            "steps": None,
            "termination": "policy_error",
            "error": f"{type(error).__name__}: {error}",
            "video": None,
        }
        (output / "result.json").write_text(json.dumps(row, indent=2) + "\n")
        print(row)
        return row

    class PredictionLog:
        def __init__(self, wrapped: ActionPolicy) -> None:
            self.wrapped = wrapped
            self.actions = []

        def reset(self) -> None:
            self.wrapped.reset()

        def select_action(self, batch: TensorBatch) -> torch.Tensor:
            action = self.wrapped.select_action(batch)
            self.actions.append(action.detach().cpu().reshape(-1).tolist())
            return action

    logged_policy = PredictionLog(policy)
    environment = None
    try:
        environment = open_libero_task(
            task, state_id, WORKSPACE / "configs", WORKSPACE / ".local-data"
        )
        result = run_policy_episode(
            environment,
            logged_policy,
            preprocessor,
            postprocessor,
            task,
            state_id,
            WORKSPACE / "configs",
            partition="development",
            max_steps=max_steps,
            on_frame=frames.append,
            task_prompt=prompt if method.startswith("smolvla") else None,
            reset_seed=20260930 + state_id,
        )
        video = encode_rgb_video(frames, output / "rollout.mp4", fps=20) if frames else None
        row = {
            "method": method,
            "state": state_id,
            "success": result.success,
            "steps": result.steps,
            "termination": result.termination_reason,
            "error": None,
            "video": str(video) if video else None,
            "normalized_predicted_actions": logged_policy.actions,
            "prompt": prompt if method.startswith("smolvla") else None,
            "checkpoint": recipe.get("checkpoint"),
        }
    except Exception as error:
        row = {
            "method": method,
            "state": state_id,
            "success": False,
            "steps": None,
            "termination": "infrastructure_error",
            "error": f"{type(error).__name__}: {error}",
            "video": None,
            "normalized_predicted_actions": logged_policy.actions,
        }
    finally:
        if environment is not None:
            environment.close()
    (output / "result.json").write_text(json.dumps(row, indent=2) + "\n")
    print({key: value for key, value in row.items() if key != "normalized_predicted_actions"})
    if row.get("video"):
        display(Video(row["video"], embed=True, mimetype="video/mp4", html_attributes="controls"))
    del logged_policy, frames, environment, policy, preprocessor, postprocessor, recipe
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return row

### LIBERO run — act

Run this cell to load the prepared checkpoint, use one development state, and display its video. The video and result record use separate files for this policy.


In [5]:
ACT_RESULT = run_one("act", 0)

Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'method': 'act', 'state': 0, 'success': True, 'steps': 74, 'termination': 'success', 'error': None, 'video': '/home/michal/code/vlas_tutorial/.local-data/course-runs/unit01/act-0-8b39ccba/rollout.mp4', 'prompt': None, 'checkpoint': '/home/michal/code/vlas_tutorial/.local-data/runs/act-bowl-plate-probe-200/checkpoint-000200'}


### LIBERO run — diffusion

Run this cell to load the prepared checkpoint, use one development state, and display its video. The video and result record use separate files for this policy.


In [7]:
DIFFUSION_RESULT = run_one("diffusion", 0)

Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'method': 'diffusion', 'state': 0, 'success': True, 'steps': 70, 'termination': 'success', 'error': None, 'video': '/home/michal/code/vlas_tutorial/.local-data/course-runs/unit01/diffusion-0-10379c22/rollout.mp4', 'prompt': None, 'checkpoint': '/home/michal/code/vlas_tutorial/.local-data/runs/diffusion-bowl-plate-reference-2000/checkpoint-002000'}


### LIBERO run — smolvla_published

Run this cell to load the prepared checkpoint, use one development state, and display its video. The video and result record use separate files for this policy.


In [9]:
SMOLVLA_RESULT = run_one("smolvla_published", 0, prompt="put the blue thing on the plate")

Loading weights from local directory
[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
{'method': 'smolvla_published', 'state': 0, 'success': False, 'steps': 160, 'termination': 'timeout', 'error': None, 'video': '/home/michal/code/vlas_tutorial/.local-data/course-runs/unit01/smolvla_published-0-ed42ab22/rollout.mp4', 'prompt': 'put the blue thing on the plate', 'checkpoint': '/home/michal/code/vlas_tutorial/.local-data/models/smolvla-libero'}


## Record one observation

For each run, write one sentence about the approach, one sentence about what worked or failed, and one question to investigate. The recorded success rows are from separate runs and do not show the same states for every policy.


In [ ]:
observations = []  # Add rows: {"method": "act", "state": 0, "note": "..."}.
observations